___
                 we're dealing with time-dependent machine trajectories.

            We don't want information from the future leaking into the past
___


use a Time-Based Split (also called a chronological or rolling split). You cut the timeline at a specific point in time
___

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler

In [2]:
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data" / "raw" / "CMAPSS"

TRAIN_FILE = DATA_DIR / "train_FD001.txt"

In [3]:
columns = [
    "engine_id",
    "cycle",
    "setting_1",
    "setting_2",
    "setting_3",
    *[f"sensor_{i}" for i in range(1, 22)]
]

In [4]:
df = pd.read_csv(
    TRAIN_FILE,
    sep=r"\s+",
    header=None,
    names=columns
)

In [5]:
df["RUL"] = (
    df.groupby("engine_id")["cycle"]
      .transform("max")
    - df["cycle"]
)

In [6]:
df[["engine_id", "cycle", "RUL"]].head()

,engine_id,cycle,RUL
0,1,1,191
1,1,2,190
2,1,3,189
3,1,4,188
4,1,5,187


## Select sensor features

In [7]:
sensor_cols = [f"sensor_{i}" for i in range(1, 22)]

In [8]:
sensor_variance = df[sensor_cols].var().sort_values()

sensor_variance

sensor_1     0.000000e+00
sensor_10    0.000000e+00
sensor_19    0.000000e+00
sensor_18    0.000000e+00
sensor_16    1.203765e-35
sensor_5     2.840037e-29
sensor_6     1.929279e-06
sensor_15    1.406628e-03
sensor_8     5.038938e-03
sensor_13    5.172330e-03
sensor_21    1.171825e-02
sensor_20    3.266927e-02
sensor_11    7.133568e-02
sensor_2     2.500533e-01
sensor_12    5.439850e-01
sensor_7     7.833883e-01
sensor_17    2.398667e+00
sensor_3     3.759099e+01
sensor_4     8.101089e+01
sensor_14    3.639005e+02
sensor_9     4.876536e+02
dtype: float64

In [9]:
constant_sensors = [
    col for col in sensor_cols
    if df[col].nunique() <= 1
]

constant_sensors

['sensor_1', 'sensor_5', 'sensor_10', 'sensor_16', 'sensor_18', 'sensor_19']

In [10]:
useful_sensors = [
    col for col in sensor_cols
    if col not in constant_sensors
]

print("Original sensors:", len(sensor_cols))
print("Useful sensors:", len(useful_sensors))
print("Removed:", constant_sensors)

Original sensors: 21
Useful sensors: 15
Removed: ['sensor_1', 'sensor_5', 'sensor_10', 'sensor_16', 'sensor_18', 'sensor_19']


Because the scaler would learn statistics from validation/test data.

That's data leakage.

### Split by engine

In [11]:
engine_ids = df["engine_id"].unique()

len(engine_ids)

100

In [12]:
from sklearn.model_selection import train_test_split

train_engines, temp_engines = train_test_split(
    engine_ids,
    test_size=0.30,
    random_state=42
)

val_engines, test_engines = train_test_split(
    temp_engines,
    test_size=0.50,
    random_state=42
)

In [13]:
print("Train engines:", len(train_engines))
print("Validation engines:", len(val_engines))
print("Test engines:", len(test_engines))

Train engines: 70
Validation engines: 15
Test engines: 15


In [14]:
train_df = df[df["engine_id"].isin(train_engines)].copy()
val_df = df[df["engine_id"].isin(val_engines)].copy()
test_df = df[df["engine_id"].isin(test_engines)].copy()

In [15]:
print(set(train_df.engine_id) & set(val_df.engine_id))
print(set(train_df.engine_id) & set(test_df.engine_id))
print(set(val_df.engine_id) & set(test_df.engine_id))

set()
set()
set()


### Fit the scaler ONLY on training data

In [16]:
scaler = StandardScaler()

train_df[useful_sensors] = scaler.fit_transform(
    train_df[useful_sensors]
)

val_df[useful_sensors] = scaler.transform(
    val_df[useful_sensors]
)

test_df[useful_sensors] = scaler.transform(
    test_df[useful_sensors]
)

### part: sequences

one row → one prediction nononnononoon

                        TIME

                        t-29
                        t-28
                        t-27
                        ...
                        t-1
                        t
                         ↓
                        LSTM / GRU
                         ↓
                        RUL

In [19]:
WINDOW_SIZE = 30

In [17]:
def create_sequences(data, feature_cols, target_col, window_size):
    X = []
    y = []

    for engine_id, engine_data in data.groupby("engine_id"):
        engine_data = engine_data.sort_values("cycle")

        features = engine_data[feature_cols].values
        targets = engine_data[target_col].values

        for i in range(window_size, len(engine_data)):
            X.append(features[i - window_size:i])
            y.append(targets[i])

    return np.array(X), np.array(y)

In [20]:
X_train, y_train = create_sequences(
    train_df,
    useful_sensors,
    "RUL",
    WINDOW_SIZE
)

X_val, y_val = create_sequences(
    val_df,
    useful_sensors,
    "RUL",
    WINDOW_SIZE
)

X_test, y_test = create_sequences(
    test_df,
    useful_sensors,
    "RUL",
    WINDOW_SIZE
)

In [21]:
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("X_val:", X_val.shape)
print("y_val:", y_val.shape)

print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

X_train: (12407, 30, 15)
y_train: (12407,)
X_val: (2612, 30, 15)
y_val: (2612,)
X_test: (2612, 30, 15)
y_test: (2612,)


In [22]:
print(X_train[0].shape)

(30, 15)


In [23]:
print(X_train[0])

[[-1.57974963 -1.08775854 -1.95433887 -7.14117748  1.3080393  -1.20350323
  -0.47502452 -2.2814432   1.23078759 -0.4852445  -0.31958655 -1.36766435
  -1.42408406  0.67289119  1.55315544]
 [-1.72086208 -0.56014387 -1.74810431  0.14003293  1.58154981 -1.62574711
  -0.6261525  -1.1166205   1.73456174 -1.59318556 -0.65860399 -0.6675596
  -0.77639972  1.34243244  1.08810056]
 [-2.26515296 -0.35139913 -1.09818673 -7.14117748  2.00321184 -0.78125935
  -0.49629767 -1.19177035  1.57117553 -1.45469293 -0.17334373 -1.64289637
  -1.42408406  1.62140796  1.24156867]
 [-2.00308698 -1.0368051  -1.41924378  0.14003293  0.9889437  -1.62574711
  -0.32921482 -1.64266946  1.44863587 -2.28564872 -0.18050247 -1.06571078
  -1.42408406  1.73299817  1.96426394]
 [-1.90229237 -1.87835872 -0.70132454 -7.14117748  1.9804193  -0.92200731
  -0.32301016 -1.07904557  1.14909449 -2.14715609 -0.37327709 -1.46920626
  -2.07176839  2.01197369  1.22668691]
 [-2.76912599 -2.12983861 -1.34343865  0.14003293  1.8208715  -0.7

In [24]:
print("Target RUL:", y_train[0])

Target RUL: 256


## RUL capping

There's another industry-style decision we need to make.

Early in a machine's life, its RUL might be:

300
299
298
297
...

But the early period may not show meaningful degradation.

In [25]:
RUL_CAP = 125

df["RUL_CAPPED"] = df["RUL"].clip(upper=RUL_CAP)

In [26]:
processed_dir = PROJECT_ROOT / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

In [27]:
np.savez_compressed(
    processed_dir / "train_sequences.npz",
    X=X_train,
    y=y_train
)

np.savez_compressed(
    processed_dir / "val_sequences.npz",
    X=X_val,
    y=y_val
)

np.savez_compressed(
    processed_dir / "test_sequences.npz",
    X=X_test,
    y=y_test
)